In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A Gaussian processis a non-parameteric tool for regression and function approximation.
It is a distribution function $f(x) \sim \mathcal{GP}(\mu(x), \sigma(x,x^\prime))$.
We start from a prior (in our case mean function $\mu(x)=0$ and covariance function $\sigma(x,x^\prime)$ defining how points are related. 

The folloiwng steps are infolved
1. define mean function
2. define covariance function
3. generate a prior distribution
4. update the prior with new data to get a posterior
5. predict new points using the posterior (aka use the model) 

define a prior on $x \in [0, 30]$

In [ ]:
N = 256
x = np.linspace(0, 30, N)

with mean value $\mu = 0$ and standard deviation $\sigma = 1$.

In [ ]:
mu_prior = np.zeros_like(x)

define correlation matrix

In [ ]:
def squared_exponential_kernel(x1, x2, l=1.0, sigma_f = 1.0):
    """
    x1 ... position 1
    x2 ... position 2
    l ... length scale
    sigma_f ... standard deviaton of mean
    """
    return sigma_f**2 * np.exp(-0.5 * ((x1-x2)/l)**2)

In [ ]:
sigma_f = 1.0 # standard deviation of signal variance
K_xx = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 1.0)

To get a random sample, that follows the mean and correlation on the domain $\mathcal{D} = [x \in \mathbb{R}; \, 0\leq x \leq 30]$, we use a "multivariate normal sample", which is drawn as follows:

 - define an vector $\vec{x}$ of sample points in $D$
 - compute the covariance matrix $K$ for this vector
 - convert the covariance matrix $K$ into a lower triangular matrix $K = L L^T$ (This is done because $K$ is symmetric and positive definite.)
 - generate a vector $\vec z$ of the same length as $\vec{x}$ that contains independent standard normal samples (mean 0, variance 1)
 - compute $\vec{s} = \mu + L \vec{z}$ to get a sample distribution $\vec{s}$ over $\vec{x}$.  

In [ ]:
# own implementation of: np.random.multivariate_normal (here only for size=1 (number of samples)
# but it easily runs into numeric instabilties 


def multivariate_normal_samples(mean, cov):
    """
    Draw samples from a multivariate normal distribution.

    Parameters:
    - mean: 1D array, mean vector of the distribution.
    - cov: 2D array, covariance matrix of the distribution.

    Returns:
    - sample: 1D array of shape len(mean)
    """
    N = len(mean)
    sample = np.zeros(N)

    # Step 1: Cholesky decomposition of the covariance matrix
    L = np.linalg.cholesky(cov)

    # Step 2: Generate standard normal samples
    z = np.random.normal(size=N)

    # Step 3: Transform to get a sample from N(mean, cov)
    sample = mean + L @ z

    return sample

### Why does this converts uncorrelated noise to exactly the smooth correlated signal (sample) we want? 

The **Cholesky factor** of the correlation matrix $K$ is defined as:
$$ \boldsymbol{K} = \boldsymbol{L} \boldsymbol{L}^T $$
with $L$ being a lower triangular matrix and $L^T$ being its transposed. 

$\vec{z}$ is a vector of  independent (uncorrelated) standard normal samples with mean 0 and variance 1:
$$ z_i \sim \mathcal{N}(0, 1) $$

As the elements of $\vec{z}$ are uncorrelated, the covariance matrix is the identity matrix $\boldsymbol{I}$:
$$ \mathrm{Cov}(\vec{z},\vec{z}) = I $$

The matrix multiplication of the Cholesky factor $\boldsymbol{L}$ with the normal sample $\vec{z}$:
$$ \vec{y} = \boldsymbol{L} \vec{z} $$
is also a vector of same length as $\vec{z}$. 

The covariance matrix of this vector is:
$$ \mathrm{Cov}(\vec{y}, \vec{y}) = \mathrm{Cov}(\boldsymbol{L} \vec{z}, \boldsymbol{L} \vec{z}) = \boldsymbol{L} \mathrm{Cov}(\vec{z}, \vec{z}) \boldsymbol{L}^T = \boldsymbol{L}\boldsymbol{I}\boldsymbol{L}^T = \boldsymbol{L}\boldsymbol{L}^T = \boldsymbol{K} $$

Thus, $\vec{y}$ has the desired covariance matrix.

In [ ]:
prior_samples = np.random.multivariate_normal(mu_prior, K_xx, 3).T

#prior_samples_own = multivariate_normal_samples(mu_prior, K_xx)

In [ ]:
plt.title(r"$\mu = 0.0$, $l = 1.0$, and $\sigma_f = 1.0$", fontsize=16)

plt.fill_between(x, mu_prior-3*sigma_f, mu_prior+3*sigma_f, color="C0", alpha=0.1)
plt.fill_between(x, mu_prior-2*sigma_f, mu_prior+2*sigma_f, color="C0", alpha=0.3)
plt.fill_between(x, mu_prior-sigma_f, mu_prior+sigma_f, color="C0", alpha=0.5)
plt.plot(x, mu_prior, color="C0")

plt.plot(x, prior_samples[:, 0], color="C1")
plt.plot(x, prior_samples[:, 1], color="C2")
plt.plot(x, prior_samples[:, 2], color="C3")
#plt.plot(x, prior_samples_own)


plt.xlabel(r"$x$", fontsize=18)
plt.xticks(fontsize=14)
plt.xlim(0,30)

plt.ylabel(r"$y$", fontsize=18)
plt.yticks(fontsize=14)
plt.ylim(-4,4)

plt.tight_layout()
plt.show()

## influence of the length scale $l$ used in the covariance matrix 

In [ ]:
plt.title(r"$\mu = 0.0$, $l = 5.0$, and $\sigma_f = 1.0$", fontsize=16)

sigma_prior = 1.0
K_xx = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 5)
prior_samples = np.random.multivariate_normal(mu_prior, K_xx, 3).T


plt.fill_between(x, mu_prior-3*sigma_f, mu_prior+3*sigma_f, color="C0", alpha=0.1)
plt.fill_between(x, mu_prior-2*sigma_f, mu_prior+2*sigma_f, color="C0", alpha=0.3)
plt.fill_between(x, mu_prior-sigma_f, mu_prior+sigma_f, color="C0", alpha=0.5)
plt.plot(x, mu_prior, color="C0")

plt.plot(x, prior_samples[:, 0], color="C1")
plt.plot(x, prior_samples[:, 1], color="C2")
plt.plot(x, prior_samples[:, 2], color="C3")
#plt.plot(x, prior_samples_own)


plt.xlabel(r"$x$", fontsize=18)
plt.xticks(fontsize=14)
plt.xlim(0,30)

plt.ylabel(r"$y$", fontsize=18)
plt.yticks(fontsize=14)
plt.ylim(-4,4)

plt.tight_layout()
plt.show()

In [ ]:
plt.title(r"$\mu = 0.0$, $l = 0.2$, and $\sigma_f = 1.0$", fontsize=16)

K_xx = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 0.2)
prior_samples = np.random.multivariate_normal(mu_prior, K_xx, 3).T


plt.fill_between(x, mu_prior-3*sigma_f, mu_prior+3*sigma_f, color="C0", alpha=0.1)
plt.fill_between(x, mu_prior-2*sigma_f, mu_prior+2*sigma_f, color="C0", alpha=0.3)
plt.fill_between(x, mu_prior-sigma_f, mu_prior+sigma_f, color="C0", alpha=0.5)
plt.plot(x, mu_prior, color="C0")

plt.plot(x, prior_samples[:, 0], color="C1")
plt.plot(x, prior_samples[:, 1], color="C2")
plt.plot(x, prior_samples[:, 2], color="C3")
#plt.plot(x, prior_samples_own)


plt.xlabel(r"$x$", fontsize=18)
plt.xticks(fontsize=14)
plt.xlim(0,30)

plt.ylabel(r"$y$", fontsize=18)
plt.yticks(fontsize=14)
plt.ylim(-4,4)

plt.tight_layout()
plt.show()

## influence of non-zero mean function $\mu(x)$

In [ ]:
plt.title(r"$\mu = \left(\frac{x-15}{10}\right)^2$, $l = 1.0$, and $sigma_f = 1.0$", fontsize=16)


mu_prior = ((x-15.0)/10.0)**2
K_xx = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 1.0)
prior_samples = np.random.multivariate_normal(mu_prior, K_xx, 3).T

plt.fill_between(x, mu_prior-3*sigma_f, mu_prior+3*sigma_f, color="C0", alpha=0.1)
plt.fill_between(x, mu_prior-2*sigma_f, mu_prior+2*sigma_f, color="C0", alpha=0.3)
plt.fill_between(x, mu_prior-sigma_f, mu_prior+sigma_f, color="C0", alpha=0.5)
plt.plot(x, mu_prior, color="C0")

plt.plot(x, prior_samples[:, 0], color="C1")
plt.plot(x, prior_samples[:, 1], color="C2")
plt.plot(x, prior_samples[:, 2], color="C3")
#plt.plot(x, prior_samples_own)


plt.xlabel(r"$x$", fontsize=18)
plt.xticks(fontsize=14)
plt.xlim(0,30)

plt.ylabel(r"$y$", fontsize=18)
plt.yticks(fontsize=14)
plt.ylim(-4,6)

plt.tight_layout()
plt.show()

## influence of signal variance $\sigma_f^2$

In [ ]:
plt.title(r"$\mu = 0.0$, $l = 1.0$, and $\sigma_f = 0.2$", fontsize=16)

mu_prior = np.zeros_like(x)
sigma_f = 0.2
K_xx = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 1.0, sigma_f)
prior_samples = np.random.multivariate_normal(mu_prior, K_xx, 3).T

plt.fill_between(x, mu_prior-3*sigma_f, mu_prior+3*sigma_f, color="C0", alpha=0.1)
plt.fill_between(x, mu_prior-2*sigma_f, mu_prior+2*sigma_f, color="C0", alpha=0.3)
plt.fill_between(x, mu_prior-sigma_f, mu_prior+sigma_f, color="C0", alpha=0.5)
plt.plot(x, mu_prior, color="C0")

plt.plot(x, prior_samples[:, 0], color="C1")
plt.plot(x, prior_samples[:, 1], color="C2")
plt.plot(x, prior_samples[:, 2], color="C3")
#plt.plot(x, prior_samples_own)


plt.xlabel(r"$x$", fontsize=18)
plt.xticks(fontsize=14)
plt.xlim(0,30)

plt.ylabel(r"$y$", fontsize=18)
plt.yticks(fontsize=14)
plt.ylim(-4,4)

plt.tight_layout()
plt.show()

In [ ]:
plt.title(r"$\mu = 0.0$, $l = 1.0$, and $\sigma_f = 1.3$", fontsize=16)

mu_prior = np.zeros_like(x)
sigma_f = 1.3
K_xx = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 1.0, sigma_f)
prior_samples = np.random.multivariate_normal(mu_prior, K_xx, 3).T

plt.fill_between(x, mu_prior-3*sigma_f, mu_prior+3*sigma_f, color="C0", alpha=0.1)
plt.fill_between(x, mu_prior-2*sigma_f, mu_prior+2*sigma_f, color="C0", alpha=0.3)
plt.fill_between(x, mu_prior-sigma_f, mu_prior+sigma_f, color="C0", alpha=0.5)
plt.plot(x, mu_prior, color="C0")

plt.plot(x, prior_samples[:, 0], color="C1")
plt.plot(x, prior_samples[:, 1], color="C2")
plt.plot(x, prior_samples[:, 2], color="C3")
#plt.plot(x, prior_samples_own)


plt.xlabel(r"$x$", fontsize=18)
plt.xticks(fontsize=14)
plt.xlim(0,30)

plt.ylabel(r"$y$", fontsize=18)
plt.yticks(fontsize=14)
plt.ylim(-4,4)

plt.tight_layout()
plt.show()